# Run `coincidence_counters_acquisition.py` (Robust version)
This notebook lets you run the acquisition script with defaults or custom arguments, run it in the background, stop it by PID, and live-plot the CSV output. It uses `subprocess` with argument lists to avoid quoting issues.

In [1]:
# 🔎 Verify the script is present next to this notebook
import os
SCRIPT = 'coincidence_counters_acquisition.py'
if not os.path.exists(SCRIPT):
    raise FileNotFoundError(f'{SCRIPT} not found in {os.getcwd()}')
print('Found script:', os.path.abspath(SCRIPT))

Found script: c:\Users\Experiment\Documents\Python\VQ_Instruments\ID1000\coincidence_counters_acquisition.py


In [2]:
# ▶️ Run with the script's default arguments
import subprocess, sys
cmd = [sys.executable, 'coincidence_counters_acquisition.py']
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)

Running: c:\Users\Experiment\Documents\Python\VQ_Instruments\instruments_win\Scripts\python.exe coincidence_counters_acquisition.py


CompletedProcess(args=['c:\\Users\\Experiment\\Documents\\Python\\VQ_Instruments\\instruments_win\\Scripts\\python.exe', 'coincidence_counters_acquisition.py'], returncode=0)

## Run with custom arguments
Adjust the parameters below, then execute the cell to run one-shot with those values.

In [ ]:
# 🛠️ Customize and run (safe quoting via list args)
import subprocess, sys
CUSTOM_INTERVAL = 2            # seconds
CUSTOM_DURATION = 20           # seconds (0 or None for infinite)
TC_ADDRESS = '169.254.224.106' # device IP
COUNTERS_FILE = 'counters_output.csv'
COINCIDENCE_WINDOW_PS = 5000   # picoseconds
INTEGRATION_NS = 1000          # nanoseconds (0 for endless accumulation)
LOG_PATH = 'acquisition_log.txt'
VERBOSE = True

cmd = [
    sys.executable, 'coincidence_counters_acquisition.py',
    '--interval', str(CUSTOM_INTERVAL),
    '--duration', str(CUSTOM_DURATION if CUSTOM_DURATION is not None else 0),
    '--address', str(TC_ADDRESS),
    '--file', str(COUNTERS_FILE),
    '--window', str(COINCIDENCE_WINDOW_PS),
    '--integration', str(INTEGRATION_NS),
    '--log-path', str(LOG_PATH),
]
if VERBOSE:
    cmd.append('--verbose')
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)

## Run in the background
Start the acquisition as a background process. The PID is saved to `acq_pid.txt` so you can stop it later—even after a kernel restart.

In [ ]:
# 🧵 Start background process
import subprocess, sys, json
BG_INTERVAL = 3
BG_DURATION = 60
BG_FILE = 'counters_output.csv'

bg_args = [
    sys.executable, 'coincidence_counters_acquisition.py',
    '--interval', str(BG_INTERVAL),
    '--duration', str(BG_DURATION),
    '--file', BG_FILE,
]
proc = subprocess.Popen(bg_args)
print('Started PID:', proc.pid)
with open('acq_pid.txt', 'w') as f:
    f.write(str(proc.pid))
print('PID saved to acq_pid.txt')

In [ ]:
# ⛔ Stop background process by PID (reads from acq_pid.txt if available)
import os, signal
pid = None
if os.path.exists('acq_pid.txt'):
    try:
        pid = int(open('acq_pid.txt').read().strip())
    except Exception:
        pid = None
if pid is None:
    print('No PID found. Set `pid = <number>` manually and re-run this cell.')
else:
    try:
        os.kill(pid, signal.SIGTERM)
        print('Sent SIGTERM to', pid)
    except ProcessLookupError:
        print('No such process:', pid)
    except PermissionError:
        print('Permission denied to stop PID', pid)

## Live plot of counters CSV (optional)
This cell refreshes periodically and plots the semicolon-separated CSV. Stop with **Kernel ▶ Interrupt** or press the stop button.

In [ ]:
# 📈 Live plotter
import os, time
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import clear_output, display

CSV_PATH = 'counters_output.csv'  # set to match your --file argument
REFRESH_EVERY = 3                 # seconds
MAX_POINTS = None                 # e.g. 500 to limit

def try_read_csv(path):
    # Handle partial writes by retrying quickly
    for _ in range(3):
        try:
            return pd.read_csv(path, sep=';')
        except Exception:
            time.sleep(0.2)
    raise

def plot_csv(path):
    if not os.path.exists(path):
        print(f'Waiting for {path} ...')
        return False
    try:
        df = try_read_csv(path)
    except Exception as e:
        print('CSV not ready yet:', e)
        return False

    if 'time' not in df.columns:
        print('CSV missing "time" column; header not written yet?')
        return False
    if MAX_POINTS:
        df = df.tail(MAX_POINTS)

    plt.figure(figsize=(10,5))
    for col in df.columns:
        if col == 'time':
            continue
        plt.plot(df['time'], df[col], label=col)
    plt.xlabel('time (s)')
    plt.ylabel('counts')
    plt.title('Coincidence & input counters vs. time')
    plt.legend(loc='best')
    plt.grid(True, alpha=0.3)
    display(plt.gcf())
    plt.close()
    return True

try:
    while True:
        clear_output(wait=True)
        plot_csv(CSV_PATH)
        time.sleep(REFRESH_EVERY)
except KeyboardInterrupt:
    print('Stopped live plotting.')